In [ ]:
# analysis
import numpy as np
import pandas as pd 

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# single-cell
import scanpy as sc
import liana as li
import maboss


BRUDNO
Basic data loading tets

In [ ]:
import maboss
import pandas as pd

# 1. Wczytanie modelu wewnątrzkomórkowego (np. sieci regulacji genów dla naszej komórki)
# Plik .bnd definiuje logikę, a .cfg parametry symulacji
sim = maboss.load("moj_model.bnd", "moj_model.cfg")

# Załóżmy, że LIANA+ dała nam takie wagi (prawdopodobieństwa) obecności sygnału w 3 punktach w czasie:
# np. stężenie cytokiny TNF w środowisku wokół naszej komórki
time_series_environment = [0.1, 0.8, 0.3] 

# 2. Pętla czasowa symulująca interakcję ze środowiskiem
czas_kroku_dt = 10 

for t, external_signal in enumerate(time_series_environment):
    print(f"--- Krok czasowy {t} ---")
    
    # a) Aktualizacja informacji ze środowiska (wymuszamy prawdopodobieństwo aktywacji receptora)
    # [prawdopodobieństwo_0, prawdopodobieństwo_1]
    sim.network.set_istate('Receptor_TNF', [1 - external_signal, external_signal])
    
    # b) Uruchomienie symulacji na czas dt
    sim.update_parameters(max_time=czas_kroku_dt)
    result = sim.run()
    
    # c) Odczytanie stanu końcowego komórki
    last_states = result.get_last_states_probtraj()
    
    # d) Magia integracji: Używamy stanu końcowego jako stanu początkowego na kolejny krok!
    # Pobieramy prawdopodobieństwa stanów węzłów z końca symulacji
    # i nadpisujemy nimi model na następny obieg pętli (z wyjątkiem receptora, który sterowany jest z zewnątrz)
    for node in sim.network.keys():
        if node != 'Receptor_TNF':
            # Uproszczenie: w prawdziwym kodzie mapujesz wyciągnięte prawdopodobieństwo z tabeli last_states
            prob_active = get_node_prob_from_last_state(last_states, node) 
            sim.network.set_istate(node, [1 - prob_active, prob_active])

    print("Symulacja kroku zakończona. Komórka zaadaptowała się do nowego środowiska.")

CZYSTO

***

### Protokół single cell

Zakładamy, że mamy wygenerwanie poprawnie macierze w formacie .mtx lub .h5ad

In [ ]:
# ----- Łączenie danych ----- #

import scanpy as sc
import scipy.sparse as sp
import anndata as ad
import numpy as np
# # W rzeczywistości wczytujemy dane z dysku np.:
# adata_ctrl = sc.read_10x_mtx('data/control/')
# adata_stim = sc.read_10x_mtx('data/stimulated/')

# Na potrzeby naszego ćwiczenia użyjemy wbudowanego zbioru i go podzielimy/zmodyfikujemy
adata_ctrl = sc.datasets.pbmc3k()
adata_ctrl.obs['condition'] = 'Control'

# Tworzymy "sztuczną" drugą próbę (np. po stymulacji lekami)
adata_stim = sc.datasets.pbmc3k()
adata_stim.obs['condition'] = 'Stimulated'
# Sztuczna zmiana dla zilustrowania zróżnic w późniejszych analizach
# UWAGA: tutaj mamy specjalny format rzadkich macierzy 
adata_stim.X = sp.csr_matrix(adata_stim.X.multiply(np.random.uniform(0.8, 1.2, size=adata_stim.X.shape)))


# ŁĄCZENIE (Concatenation) - to najważniejszy krok przy wielu próbach
# outer join zapewnia, że jeśli w jednej próbie gen nie uległ ekspresji, zostanie wypełniony zerami
adata = ad.concat([adata_ctrl, adata_stim], label='batch', keys=['ctrl', 'stim'], join='outer')
adata.obs_names_make_unique() # Zabezpieczenie przed dublowaniem się kodów kreskowych komórek

print(adata_ctrl)
print(adata)
# Zobaczysz wymiary: n_cells x n_genes oraz zmienne w adata.obs: 'condition', 'batch'

In [ ]:
# ---- Kontrola jakości ---- # 
# 1. Oznaczenie genów mitochondrialnych (i opcjonalnie rybosomalnych)
# W przypadku ludzkich genów zaczynają się one od 'MT-'
adata.var['mt'] = adata.var_names.str.startswith('MT-')
# W przypadku myszy byłoby to 'mt-'

# 2. Obliczenie statystyk QC
# scanpy automatycznie obliczy n_genes_by_counts, total_counts i pct_counts_mt
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# 3. Zapiszmy stan przed filtrowaniem, żeby zobaczyć, ile odrzuciliśmy
n_cells_raw = adata.n_obs

# 4. Filtrowanie (Twarde progi - w prawdziwych badaniach dobierasz je na podstawie wykresów)
# Minimalna liczba genów (usuwa puste krople i szczątki komórkowe)
sc.pp.filter_cells(adata, min_genes=200)

# Maksymalna liczba genów (usuwa potencjalne dublety - komórki o podejrzanie bogatym transkryptomie)
adata = adata[adata.obs.n_genes_by_counts < 2500, :]

# Frakcja mitochondrialna (zwykle 5-10% dla zdrowych PBMC, ale dla guzów może to być nawet 20-30%)
adata = adata[adata.obs.pct_counts_mt < 5, :]

# 5. Filtrowanie genów
# Usuwamy geny, które ulegają ekspresji w mniej niż np. 3 komórkach (totalny szum, nie przydadzą się do żadnej statystyki)
sc.pp.filter_genes(adata, min_cells=3)

print(f"Liczba komórek przed: {n_cells_raw}")
print(f"Liczba komórek po QC: {adata.n_obs}")

In [ ]:
# ---- Normalizacja oraz wykrywanie wysoce zmiennych genów ---- # 

# 1. Zachowanie surowych danych (Raw counts)
# Bardzo ważne: LIANA+ i testy różnicowej ekspresji często wymagają surowych zliczeń całkowitych!
adata.raw = adata.copy()

# 2. Normalizacja (do 10,000 zliczeń na komórkę)
sc.pp.normalize_total(adata, target_sum=1e4)

# 3. Logarytmizacja
sc.pp.log1p(adata)

# 4. Identyfikacja HVG
# Używamy metody 'seurat' (wymaga danych zlogarytmowanych) lub 'cell_ranger'
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

# Wizualizacja HVG (opcjonalnie)
# sc.pl.highly_variable_genes(adata)

# 5. Filtrowanie macierzy do HVG
# Robimy kopię lub pracujemy na widoku, aby nie stracić reszty genów
adata_hvg = adata[:, adata.var.highly_variable].copy()

In [ ]:
# ---- Wycieszenie genów mitochondrioalnych żeby nie wprowadzały szumu ----- 
# Regresja efektów technicznych
sc.pp.regress_out(adata_hvg, ['total_counts', 'pct_counts_mt'])

# Skalowanie (Unit variance, zero mean)
sc.pp.scale(adata_hvg, max_value=10)

In [ ]:
# ---- redukcja wymiarowości i redukcja grafu ----
# 1. PCA
# n_comps określa liczbę wymiarów, które zachowujemy (zwykle 30-50)
sc.tl.pca(adata_hvg, n_comps=50, svd_solver='arpack')

# 2. Budowa grafu sąsiedztwa (KNN)
# n_neighbors to liczba sąsiadów (zwykle 10-15). Większa liczba = bardziej globalna struktura.
# use_rep='X_pca' mówi algorytmowi, by patrzył na składowe PCA, a nie na surowe geny.
sc.pp.neighbors(adata_hvg, n_neighbors=15, n_pcs=40)

# 3. Klastrowanie LEIDEN
# resolution=1.0 to standardowy start. Zmieniamy go zależnie od potrzebnej ziarnistości.
# flavor='vtraag' lub 'igraph'
sc.tl.leiden(adata_hvg, resolution=1.0, key_added='leiden_res_1', flavor='igraph')

# 4. Wizualizacja UMAP (Uniform Manifold Approximation and Projection)
# UMAP służy tylko do wizualizacji grafu w 2D, nie używamy go do klastrowania! - dlaczego ? 
sc.tl.umap(adata_hvg)

# Wizualizacja wyników
sc.pl.umap(adata_hvg, color=['leiden_res_1', 'condition'], wspace=0.4)

In [ ]:
adata

In [ ]:
adata_hvg

Liana part

In [ ]:
import liana as li
import plotnine as p9

In [ ]:
# Assume your object is already loaded and preprocessed as described
adata = adata_hvg.copy()

# Ensure we have sensible cell labels (rename if needed)
print(adata.obs['leiden_res_1'].value_counts())
# If you have better annotations, use those instead of leiden_res_1

In [ ]:
# Run consensus (recommended starting point)
# here we use all method and we get statistic that tell us about how valid certain method is 
li.mt.rank_aggregate(
    adata,
    groupby='leiden_res_1',      # cell groups
    resource_name='consensus',   # rich, high-quality resource
    expr_prop=0.1,               # filter low-expression LR pairs
    use_raw=True,                # use .raw (recommended for LR)
    key_added='liana_rank',      # store in adata.uns as df (in dict manner) we can get it here
    inplace=True,
    verbose=True
)

# Results are in adata.uns['liana_rank']
res = adata.uns['liana_rank']
res
# Columns include: source, target, ligand_complex, receptor_complex,
# magnitude_rank, specificity_rank, etc. + scores from individual methods

In [ ]:
# Example: CellPhoneDB (good for specificity via permutations)
# here we run only single method (for comparison)
li.mt.cellphonedb(
    adata,
    groupby='leiden_res_1',
    resource_name='consensus',
    expr_prop=0.1,
    n_perms=1000,      # more perms = better p-value resolution (slower)
    key_added='cpdb_res',
    inplace=True
)

adata.uns['cpdb_res']

In [ ]:
res.columns

In [ ]:
# Top specific & strong interactions
top_res = res.sort_values(['magnitude_rank', 'specificity_rank']).head(50)

# Filter
filtered = res[
    (res['lr_means'] > 0.1) &
    (res['magnitude_rank'] > 0.1) &
    (res['specificity_rank'] < 0.05)   # or whatever column your method uses
]

filtered

In [ ]:
adata.uns['cpdb_res']

DO zrobienia
- przeczytać co robią konkretne parametry i jakie dosajemy wartości
- zadać jakieśfajne pytanie,- jak silna jest dana ekspresja, i jakie problemy możęmy tutaj analizować
- na ile istotna dana statystyka jest dla danego problemu ?? - trzeba jakiś problem sobie dać (jak z badań) żeby zrozumieć te metody 
- 

In [ ]:
# === 1. Dotplot – najsilniejsze i najbardziej specyficzne interakcje ===
li.pl.dotplot(
    adata=adata,
    uns_key='liana_rank',         # <-- tutaj był błąd!
    colour='magnitude_rank',       # siła (magnitude)
    size='specificity_rank',       # specyficzność
    inverse_size=True,             # mały rank → duży punkt
    inverse_colour=True,           # mały rank → ciemniejszy kolor
    top_n=20,
    orderby='magnitude_rank',      # <-- OBOWIĄZKOWE!
    orderby_ascending=True,        # niższy rank = lepszy
    filter_fun=lambda x: x['specificity_rank'] < 0.1,   # możesz zostawić lub podnieść do 0.05
    source_labels=None,
    target_labels=None,
    figure_size=(10, 7)
) + p9.theme_minimal() + p9.theme(axis_text_x=p9.element_text(rotation=90))

In [ ]:
# === 2. Tileplot – diagnostyka ekspresji (means + props) ===
# (działa, bo rank_aggregate zostawia kolumny lr_means i props)
li.pl.tileplot(
    adata=adata,
    uns_key='liana_rank',
    fill='lr_means',          # średnia ekspresja LR
    label='lr_means',            # tutaj po czym będziemy porównywać
    top_n=15,
    orderby='specificity_rank',
    orderby_ascending=True
)

In [ ]:
conditions = adata.obs['condition'].unique()

for cond in conditions:
    sub = adata[adata.obs['condition'] == cond].copy()
    li.mt.rank_aggregate(sub, groupby='leiden_res_1', resource_name='consensus', 
                         expr_prop=0.1, key_added=f'liana_{cond}')
    # Compare results manually or plot with dotplot_by_sample

In [ ]:
import decoupler as dc
import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# 1. Tworzymy "pełny" AnnData z surowymi counts z .raw
adata_counts = adata.raw.to_adata()           # .X = to co było w .raw
adata_counts.obs = adata.obs.copy()           # przenosimy condition, batch, leiden_res_1
# (opcjonalnie) adata_counts.var = adata.raw.var.copy()

# Sprawdź czy counts są liczbami całkowitymi (bardzo ważne dla PyDESeq2!)
print("Min/Max w .X (powinno być >=0 i integer):", 
      adata_counts.X.min(), adata_counts.X.max())
print("Typ danych:", adata_counts.X.dtype)

# 2. Pseudobulk – teraz raw=False, bo .X już jest counts
pdata = dc.pp.pseudobulk(
    adata_counts,
    sample_col='batch',
    groups_col='leiden_res_1',
    raw=False,                    # KLUCZOWE
    mode='sum',
    skip_checks=False
)

print("Pseudobulk gotowy – wymiary:", pdata.shape)
# ... (po udanym pseudobulk powyżej)

dea_results = {}
reference_condition = 'control'      # ← ZMIEŃ na dokładną nazwę Twojego reference
test_condition      = 'stimulated'   # ← ZMIEŃ na nazwę test condition

for ct in pdata.obs['leiden_res_1'].unique():
    ctdata = pdata[pdata.obs['leiden_res_1'] == ct].copy()
    
    genes = dc.pp.filter_by_expr(ctdata, group='condition', min_count=5)
    ctdata = ctdata[:, genes].copy()
    
    dds = DeseqDataSet(
        adata=ctdata,
        design_factors='condition',
        ref_level=['condition', reference_condition]
    )
    dds.deseq2()
    stat_res = DeseqStats(dds, contrast=['condition', test_condition, reference_condition])
    stat_res.summary()
    stat_res.lfc_shrink()
    dea_results[ct] = stat_res.results_df

dea_df = pd.concat(dea_results, names=['leiden_res_1']).reset_index()

# Mapowanie na interakcje LR
lr_diff = li.multi.df_to_lr(
    adata=adata,                     # oryginalny (HVG) jest OK tutaj
    dea_df=dea_df,
    resource_name='consensus',
    stat_keys=['stat', 'pvalue', 'padj', 'log2FoldChange'],
    groupby='leiden_res_1',
    expr_prop=0.1,
    use_raw=True
)

# Wizualizacja różnic
li.pl.dotplot(
    liana_res=lr_diff,
    colour='interaction_stat',
    size='interaction_padj',
    inverse_size=True,
    top_n=20,
    orderby='interaction_stat',
    orderby_ascending=False,
    orderby_absolute=True,
    figure_size=(12, 7)
) + p9.theme_bw() + p9.scale_color_cmap('RdBu_r', midpoint=0)

In [ ]:
import decoupler as dc
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import pandas as pd

# 1. Pseudobulk – użyj raw=True (bierze adata.raw.X – tam są counts / log1p)
pdata = dc.pp.pseudobulk(
    adata,
    sample_col='batch',          # Twoje próbki / replikaty
    groups_col='leiden_res_1',   # typy komórek
    raw=True,                    # <--- KLUCZOWE!
    layer=None,                  # nie używamy layer, tylko .raw
    mode='sum',
    skip_checks=False
)

# 2. DEA per cell type (dla każdego typu komórek osobno)
dea_results = {}
reference_condition = 'control'      # <-- ZMIEŃ na nazwę Twojego reference (np. 'WT', 'ctrl')
test_condition      = 'stimulated'   # <-- ZMIEŃ na nazwę Twojego test condition

for ct in pdata.obs['leiden_res_1'].unique():
    ctdata = pdata[pdata.obs['leiden_res_1'] == ct].copy()
    
    # Filtrujemy niskiej jakości geny
    genes = dc.pp.filter_by_expr(ctdata, group='condition', min_count=5)
    ctdata = ctdata[:, genes].copy()
    
    # DESeq2
    dds = DeseqDataSet(
        adata=ctdata,
        design_factors='condition',
        ref_level=['condition', reference_condition]
    )
    dds.deseq2()
    
    stat_res = DeseqStats(dds, contrast=['condition', test_condition, reference_condition])
    stat_res.summary()
    stat_res.lfc_shrink()                    # shrinkage LFC (lepsze wartości)
    
    dea_results[ct] = stat_res.results_df

# Łączymy wszystko w jeden DataFrame
dea_df = pd.concat(dea_results, names=['leiden_res_1']).reset_index()

# 3. Mapujemy statystyki genowe na interakcje LR
lr_diff = li.multi.df_to_lr(
    adata=adata,
    dea_df=dea_df,
    resource_name='consensus',
    stat_keys=['stat', 'pvalue', 'padj', 'log2FoldChange'],   # co chcemy zachować
    groupby='leiden_res_1',
    expr_prop=0.1,
    use_raw=True
)

# 4. Wizualizacja różnic między warunkami
li.pl.dotplot(
    liana_res=lr_diff,                    # <-- tu podajemy gotowy DataFrame, nie adata
    colour='interaction_stat',            # dodatnia = upregulated w test_condition
    size='interaction_padj',
    inverse_size=True,
    top_n=15,
    orderby='interaction_stat',
    orderby_ascending=False,              # największe zmiany na górze
    orderby_absolute=True,
    figure_size=(11, 6)
) + p9.theme_bw() + p9.scale_color_cmap('RdBu_r', midpoint=0)

BRUDNO (poprzednia wersja)

In [ ]:

# ---- porządkowanie wielu agregacji ----
import liana as li

# 1. Podstawowa analiza CCC dla całego zbioru
# Metoda rank_aggregate wykonuje konsensus wielu metod
li.mt.rank_aggregate(
    adata, 
    groupby='leiden', 
    resource_name='consensus', # Używa ujednoliconej bazy OmniPath
    expr_prop=0.1,             # Minimum 10% komórek w klastrze musi mieć ekspresję L lub R
    verbose=True
)

# Wyniki są zapisane w adata.uns['liana_res']
df_res = adata.uns['liana_res']

# 2. Porównanie warunków (Control vs Stimulated)
# LIANA+ pozwala na łatwe filtrowanie wyników per warunek
li.mt.rank_aggregate(
    adata,
    groupby='leiden',
    base_iterable=adata.obs['condition'].unique(), # Iteruje po warunkach
    inplace=True
)